In [1]:
import sys
import torch
import pickle
import os
from tqdm.notebook import tqdm

sys.path.insert(0, '..')
sys.path.insert(0, '../../')
sys.path.insert(0, '../../../')
sys.path.insert(0, '../../../../')
sys.path.insert(0, '../../../../../')
sys.path.insert(0, '../../../../../../')

from stochasticLSTM.model import StochasticLSTMWeytjens
from robustness.weytjens_evaluation import evaluate_with_predefined_prefixes


In [7]:
# Load model (two instances: stochastic for MC sampling, deterministic for point estimate)
file_path_model = '../../notebooks/training_variational_dropout/Helpdesk/Helpdesk_weytjens.pkl'
output_dir = '../../../../../evaluation_results/weytjens/helpdesk/last_event_attack_all/'

model = StochasticLSTMWeytjens.load(file_path_model, p_fix=0.05)
model_without_drop = StochasticLSTMWeytjens.load(file_path_model, p_fix=0)

# Load datasets
# Note: the Helpdesk dataset uses 'Activity' as the activity column name
file_path_original = '../../../../../encoded_data/weytjens/helpdesk/helpdesk_all_5_test.pkl'
file_path_redo_activity = '../../../../../encoded_data/weytjens/helpdesk/val_new.pkl'
file_path_redo_activity_pert = '../../../../../encoded_data/weytjens/helpdesk/val_new.pkl'

original_dataset = torch.load(file_path_original, weights_only=False)
redo_activity_dataset = torch.load(file_path_redo_activity, weights_only=False)
redo_activity_pert_dataset = torch.load(file_path_redo_activity_pert, weights_only=False)

print(f'Original dataset loaded: {len(original_dataset)} cases')
print(f'Clean pairs loaded: {len(redo_activity_dataset)} pairs')
print(f'Perturbed pairs loaded: {len(redo_activity_pert_dataset)} pairs')


Data set categories:  ([('Activity', 16, {'Assign seriousness': 1, 'Closed': 2, 'Create SW anomaly': 3, 'DUPLICATE': 4, 'EOS': 5, 'INVALID': 6, 'Insert ticket': 7, 'RESOLVED': 8, 'Require upgrade': 9, 'Resolve SW anomaly': 10, 'Resolve ticket': 11, 'Schedule intervention': 12, 'Take in charge ticket': 13, 'VERIFIED': 14, 'Wait': 15})], [('case_elapsed_time', 1, {})])
Model input features:  [['Activity'], ['case_elapsed_time']]


Embeddings:  ModuleList(
  (0): Embedding(16, 8)
)
Total embedding feature size:  8
Input feature size:  9
Cells hidden size:  10
Number of LSTM layer:  2
Dropout rate:  0.05


Output feature list of dicts (featue name, tensor index in dataset):  {'case_elapsed_time': 0}
Data set categories:  ([('Activity', 16, {'Assign seriousness': 1, 'Closed': 2, 'Create SW anomaly': 3, 'DUPLICATE': 4, 'EOS': 5, 'INVALID': 6, 'Insert ticket': 7, 'RESOLVED': 8, 'Require upgrade': 9, 'Resolve SW anomaly': 10, 'Resolve ticket': 11, 'Schedule intervention': 12, 'Take in charge t

In [8]:
def save_chunk(results, i, output_dir):
    """Save intermediate results to a numbered chunk file."""
    chunk_number = (i + 1)
    filename = os.path.join(output_dir, f'robustness_results_part_{chunk_number:03d}.pkl')
    with open(filename, 'wb') as f:
        pickle.dump(results, f)
    print(f'Saved {len(results)} results to {filename}')


In [9]:
os.makedirs(output_dir, exist_ok=True)

# Create evaluation generators for clean and perturbed prefix-suffix pairs.
# The Helpdesk dataset uses 'Activity' as the activity column name.
evaluate_with_predefined_prefixes_normal = evaluate_with_predefined_prefixes(
    model=model,
    model_without_drop=model_without_drop,
    dataset=original_dataset,
    predefined_pairs=redo_activity_dataset,
    device=torch.device('cpu'),
    samples_per_case=100,
    random_order=False,
    concept_name='Activity',
)

evaluate_with_predefined_prefixes_pert = evaluate_with_predefined_prefixes(
    model=model,
    model_without_drop=model_without_drop,
    dataset=original_dataset,
    predefined_pairs=redo_activity_pert_dataset,
    device=torch.device('cpu'),
    samples_per_case=100,
    random_order=False,
    concept_name='Activity',
)

print('Evaluation generators created')


Evaluation generators created


In [10]:
# Main evaluation loop
save_every = 50
results = {}

for i, ((case_name_orig, prefix_len_orig, prefix_orig, sampled_cets_orig, suffix_orig, mean_cet_orig),
        (case_name_pert, prefix_len_pert, prefix_pert, sampled_cets_pert, suffix_pert, mean_cet_pert)) in enumerate(
        tqdm(zip(evaluate_with_predefined_prefixes_normal, evaluate_with_predefined_prefixes_pert),
             desc='Evaluating robustness')):

    key = (case_name_orig, prefix_len_orig)
    results[key] = {
        'original': (prefix_orig, suffix_orig, mean_cet_orig, sampled_cets_orig),
        'perturbed': (prefix_pert, suffix_pert, mean_cet_pert, sampled_cets_pert)
    }

    if (i + 1) % save_every == 0:
        save_chunk(results, i, output_dir)
        results = {}

if len(results):
    save_chunk(results, i, output_dir)

print('Robustness evaluation completed!')


Evaluating robustness: 0it [00:00, ?it/s]

  0%|          | 0/1898 [00:00<?, ?it/s]

RuntimeError: The size of tensor a (12) must match the size of tensor b (9) at non-singleton dimension 1

In [ ]:
# Load all saved chunks and combine them into a single results file
all_results = {}
chunk_files = sorted([f for f in os.listdir(output_dir) if f.startswith('robustness_results_part_')])

print(f'Found {len(chunk_files)} chunk files')

for chunk_file in chunk_files:
    chunk_path = os.path.join(output_dir, chunk_file)
    print(f'Loading {chunk_file}...')
    with open(chunk_path, 'rb') as f:
        chunk_results = pickle.load(f)
        all_results.update(chunk_results)
        print(f'  Added {len(chunk_results)} results from {chunk_file}')

if 'results' in locals() and len(results) > 0:
    print(f'Adding final {len(results)} results...')
    all_results.update(results)

print(f'\nTotal results loaded: {len(all_results)}')

combined_results_path = os.path.join(output_dir, 'robustness_results.pkl')
with open(combined_results_path, 'wb') as f:
    pickle.dump(all_results, f)

print(f'Combined results saved to {combined_results_path}')
